In [54]:
import pandas as pd
import numpy as np

df = pd.read_csv('student_admission_record_dirty.csv')

print("Shape:", df.shape)
print("\nNulls per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:\n", df.dtypes)

Shape: (157, 7)

Nulls per column:
 Name                      10
Age                       10
Gender                    10
Admission Test Score      11
High School Percentage    11
City                      10
Admission Status          10
dtype: int64

Duplicate rows: 6

Data types:
 Name                          str
Age                       float64
Gender                        str
Admission Test Score      float64
High School Percentage    float64
City                          str
Admission Status              str
dtype: object


DATA REPORT
The dataset has 157 rows and 7 columns. Every column has some missing data, around 10-11 missing values each. There are also 6 rows that are exact duplicates. Age, Admission Test Score, and High School Percentage are numbers, while the rest are text. Age shows up as a float instead of a whole number because of the missing values - pandas does that automatically.

In [55]:
print("Gender values:", df['Gender'].unique())
print("City values:", df['City'].unique())
print("Admission Status values:", df['Admission Status'].unique())
print(df['City'].apply(lambda x: repr(x)).unique())

Gender values: <StringArray>
['Female', 'Male', nan]
Length: 3, dtype: str
City values: <StringArray>
[    'Quetta',    'Karachi',  'Islamabad',     'Lahore',     'Multan',
          nan,   'Peshawar', 'Rawalpindi']
Length: 8, dtype: str
Admission Status values: <StringArray>
['Rejected', nan, 'Accepted']
Length: 3, dtype: str
<StringArray>
[    ''Quetta'',    ''Karachi'',  ''Islamabad'',     ''Lahore'',
     ''Multan'',          'nan',   ''Peshawar'', ''Rawalpindi'']
Length: 8, dtype: str


Standardisation Check

Checked Gender, City, and Admission Status for inconsistent formatting (different capitalization, extra spaces, etc). All values are already clean and consistent, so no standardisation was needed for these columns.

In [56]:
# Drop rows where Name or Admission Status is missing
df = df.dropna(subset=['Name', 'Admission Status'])

# Fill numeric columns with median
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Admission Test Score'] = df['Admission Test Score'].fillna(df['Admission Test Score'].median())
df['High School Percentage'] = df['High School Percentage'].fillna(df['High School Percentage'].median())

# Fill categorical columns with mode
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['City'] = df['City'].fillna(df['City'].mode()[0])

print(df.isnull().sum())

Name                      0
Age                       0
Gender                    0
Admission Test Score      0
High School Percentage    0
City                      0
Admission Status          0
dtype: int64


Missing Data Handling

Name and Admission Status had missing values, and both were handled by removing those rows — you can't guess a name, and Admission Status is the outcome column, so faking that value would create bad data.

Age, Admission Test Score, and High School Percentage are numeric, so missing values were filled with the median of each column instead of the mean, since median isn't thrown off by outliers.

Gender and City are categories, so missing values were filled with the mode (most common value) of each column.

In [57]:
#removing duplicates
print("Duplicates before removing:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after removing:", df.duplicated().sum())
print("New shape:", df.shape)

Duplicates before removing: 7
Duplicates after removing: 0
New shape: (132, 7)


Duplicate Removal

Found 7 fully duplicate rows and removed them. Row count went from 139 to 132 after removing duplicates (following the missing-data cleanup).

In [58]:
# Check for outliers in numeric columns using IQR
for col in ['Age', 'Admission Test Score', 'High School Percentage']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers, valid range [{lower:.2f}, {upper:.2f}]")
    print(outliers[[col]])
    print()

Age: 3 outliers, valid range [12.00, 28.00]
     Age
50  -1.0
113 -1.0
155 -1.0

Admission Test Score: 2 outliers, valid range [44.00, 116.00]
    Admission Test Score
31                  -5.0
91                 150.0

High School Percentage: 1 outliers, valid range [37.48, 116.15]
    High School Percentage
10                   -10.0



In [59]:
# Replace impossible negative values with NaN
df.loc[df['Age'] < 0, 'Age'] = np.nan
df.loc[df['Admission Test Score'] < 0, 'Admission Test Score'] = np.nan
df.loc[df['High School Percentage'] < 0, 'High School Percentage'] = np.nan

# Re-impute with median
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Admission Test Score'] = df['Admission Test Score'].fillna(df['Admission Test Score'].median())
df['High School Percentage'] = df['High School Percentage'].fillna(df['High School Percentage'].median())

print(df.isnull().sum())
print(df[['Age','Admission Test Score','High School Percentage']].describe())

Name                      0
Age                       0
Gender                    0
Admission Test Score      0
High School Percentage    0
City                      0
Admission Status          0
dtype: int64
              Age  Admission Test Score  High School Percentage
count  132.000000            132.000000              132.000000
mean    20.393939             78.363636               76.524621
std      2.274175             14.923736               13.937295
min     17.000000             50.000000               40.700000
25%     19.000000             71.000000               67.075000
50%     20.000000             78.500000               77.690000
75%     22.000000             89.000000               86.647500
max     24.000000            150.000000              110.500000


Outlier Detection

Used the IQR method to check Age, Admission Test Score, and High School Percentage for outliers. Found negative values in all three (like Age = -1, High School Percentage = -10), which are impossible in real life, so these were treated as data entry errors, replaced with NaN, and re-filled using the median.

One remaining outlier is a Test Score of 150 - since scoring scale wasn't specified (might not be out of 100), this was left as-is rather than assumed to be an error.

In [60]:
print(df.dtypes)

Name                          str
Age                       float64
Gender                        str
Admission Test Score      float64
High School Percentage    float64
City                          str
Admission Status              str
dtype: object


Data Type Correction

Checked data types after cleaning. Age, Admission Test Score, and High School Percentage are numeric (float), and the rest are text — all correct for this dataset, so no changes were needed.

In [61]:
# Reload the original file fresh to get "before" stats
df_original = pd.read_csv('student_admission_record_dirty.csv')

summary = pd.DataFrame({
    'Metric': ['Row Count', 'Null Count', 'Duplicate Count'],
    'Before': [
        df_original.shape[0],
        df_original.isnull().sum().sum(),
        df_original.duplicated().sum()
    ],
    'After': [
        df.shape[0],
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ]
})
print(summary)

            Metric  Before  After
0        Row Count     157    132
1       Null Count      72      0
2  Duplicate Count       6      0


Before vs After Summary

Row count went from 157 to 132 after removing rows with missing Name/Admission Status and duplicate rows. Nulls went from 72 down to 0. Duplicates went from 6 to 0. The dataset is now fully clean and ready for analysis.

In [62]:
df.to_csv('student_admission_record_cleaned.csv', index=False)